## Climate Zones - Data Preparation
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

This notebook is not a core part of the tutorial, but rather to document the process of how the dataset was prepared for the tutorial from the original dataset.

In [1]:
import pathlib
import os
import json

In [2]:
import matplotlib.pyplot
import cartopy.crs

In [3]:
import xarray
import pandas

#### Dataset parameters

In [4]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, very cold winter',
  'Dwa': 'Cold, dry winter, hot summer',
  'Dwb': 'Cold, dry winter, warm summer',
  'Dwc': 'Cold, dry winter, cold su

In [5]:
def get_platform_dir(select_platform):
    if select_platform == 'mo_linux':
      return pathlib.Path(os.environ['DATADIR']) / 'climate_zones'
    if select_platform == 'jasmin':
        return pathlib.Path('/gws/nopw/j04/mohc_shared/dscop/') / 'climate_zones'
    print('platform not found, return generic path')
    return pathlib.Path(os.environ['HOME']) / 'climate_zones',

current_platform = tutorial_config['platform']

In [6]:
root_data_dir = get_platform_dir(current_platform)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/data/users/stephen.haddad/climate_zones')

In [7]:
ml_ready_output_dir = root_data_dir / 'ml_ready'

if ml_ready_output_dir.is_dir():
    print(f'output directory {str(ml_ready_output_dir)} exists')
else:
    ml_ready_output_dir.mkdir(parents=False)
    print(f'created output directory {str(ml_ready_output_dir)}')    
    

output directory /data/users/stephen.haddad/climate_zones/ml_ready exists


In [8]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [9]:
time_periods

{(1901, 1930): ['historic'],
 (1931, 1960): ['historic'],
 (1961, 1990): ['historic'],
 (1991, 2020): ['historic'],
 (2041, 2070): ['ssp119',
  'ssp126',
  'ssp245',
  'ssp370',
  'ssp434',
  'ssp460',
  'ssp585'],
 (2071, 2099): ['ssp119',
  'ssp126',
  'ssp245',
  'ssp370',
  'ssp434',
  'ssp460',
  'ssp585']}

In [10]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
csv_out_template = tutorial_config['csv_out_template']
period_scenario_template = 'climate_zones_{start}_{end}_{scenario}_{res}.csv'

The following lables for climate zones are taken from the `legends.txt` file in the [dataset](https://figshare.com/articles/dataset/High-resolution_1_km_K_ppen-Geiger_maps_for_1901_2099_based_on_constrained_CMIP6_projections/21789074)

In [11]:
# legend_path = root_data_dir / 'legend.txt'
# with open(legend_path) as f1:
#     zones_legend = f1.readlines()
# print(''.join(zones_legend))


In [12]:
climate_subgroups_dict = tutorial_config['climate_subgroups']#
climate_subgroups_dict

{'non-land': 'None',
 'Af': 'Tropical, rainforest',
 'Am': 'Tropical, monsoon',
 'Aw': 'Tropical, savannah',
 'BWh': 'Arid, desert, hot',
 'BWk': 'Arid, desert, cold',
 'BSh': 'Arid, steppe, hot',
 'BSk': 'Arid, steppe, cold',
 'Csa': 'Temperate, dry summer, hot summer',
 'Csb': 'Temperate, dry summer, warm summer',
 'Csc': 'Temperate, dry summer, cold summer',
 'Cwa': 'Temperate, dry winter, hot summer',
 'Cwb': 'Temperate, dry winter, warm summer',
 'Cwc': 'Temperate, dry winter, cold summer',
 'Cfa': 'Temperate, no dry season, hot summer',
 'Cfb': 'Temperate, no dry season, warm summer',
 'Cfc': 'Temperate, no dry season, cold summer',
 'Dsa': 'Cold, dry summer, hot summer',
 'Dsb': 'Cold, dry summer, warm summer',
 'Dsc': 'Cold, dry summer, cold summer',
 'Dsd': 'Cold, dry summer, very cold winter',
 'Dwa': 'Cold, dry winter, hot summer',
 'Dwb': 'Cold, dry winter, warm summer',
 'Dwc': 'Cold, dry winter, cold summer',
 'Dwd': 'Cold, dry winter, very cold winter',
 'Dfa': 'Cold, no

In [13]:
climate_subgroup_lookup = [k1 for k1 in climate_subgroups_dict.keys()]

In [14]:
climate_subgroup_lookup

['non-land',
 'Af',
 'Am',
 'Aw',
 'BWh',
 'BWk',
 'BSh',
 'BSk',
 'Csa',
 'Csb',
 'Csc',
 'Cwa',
 'Cwb',
 'Cwc',
 'Cfa',
 'Cfb',
 'Cfc',
 'Dsa',
 'Dsb',
 'Dsc',
 'Dsd',
 'Dwa',
 'Dwb',
 'Dwc',
 'Dwd',
 'Dfa',
 'Dfb',
 'Dfc',
 'Dfd',
 'ET',
 'EF']

For comparison we will try predicting all 30 zones, but also trying to predict the five zone groupings for a simpler problem.

In [15]:
climate_group_lookup = [k1[0] for k1 in climate_subgroups_dict.keys()]
climate_group_lookup[0] = 'none'
climate_group_lookup

['none',
 'A',
 'A',
 'A',
 'B',
 'B',
 'B',
 'B',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E']

### Load the data

In [16]:
def get_data_path(root_dir, time_period, scenario_id, prefix, resolution_str, suffix) :
    start_year = time_period[0]
    end_year = time_period[1]
    if scenario_id == historic_scenario_str:
        data_dir = root_data_dir / time_dir_template.format(start_year=start_year,end_year=end_year)
    else:
        data_dir = root_data_dir / time_dir_template.format(start_year=start_year,end_year=end_year) / scenario_id 
    data_fname = fname_template.format(prefix=prefix, 
                                       res=resolution_str, 
                                       suffix=format_str)
    return data_dir / data_fname

In [17]:
data_path_dict = {
    (start_year, end_year): {
        scenario_id: { ds_id: { current_res: get_data_path(root_dir=root_data_dir,
                                                           time_period=(start_year, end_year),
                                                           scenario_id=scenario_id, 
                                                           prefix=ds_str,
                                                           resolution_str=res_str,
                                                           suffix=format_str,
                                                          )
                                for current_res, res_str in resolutions_dict.items() 
                              }
                       for ds_id, ds_str in dataset_prefix_dict.items()
                     } 
        for scenario_id in current_scenarios
    }
    for (start_year, end_year), current_scenarios in time_periods.items()                                                                                                             
}

In [18]:
data_path_dict

{(1901,
  1930): {'historic': {'climate_mean': {0.1: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_mean_0p1.nc'),
    0.5: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_mean_0p5.nc'),
    1.0: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_mean_1p0.nc')},
   'climate_std': {0.1: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_std_0p1.nc'),
    0.5: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_std_0p5.nc'),
    1.0: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/ensemble_std_1p0.nc')},
   'climate_zone': {0.1: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/koppen_geiger_0p1.nc'),
    0.5: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/koppen_geiger_0p5.nc'),
    1.0: PosixPath('/data/users/stephen.haddad/climate_zones/1901_1930/koppen_geiger_1p0.nc')}}},
 (1931,
  1960): {'historic': {'climate_mean': {0.1: PosixPath('/dat

### Preparing a tabulated dataset
Currently the data is in gridded format in the netcdf files. For our simple machine learning model, we are going to be predicting the climtae zone for a particular point for 

We'll start by loading the three parts to the data for the historical period 1901 to 1930 and also for a future period 2017 to 2099. We load both of these because the data is lsightly different. For the historical period we have 1 simulation, whereas for the future period, we have different climate change scenarios, called [Shared Socioeconomic Pathways](https://en.wikipedia.org/wiki/Shared_Socioeconomic_Pathways), each represented in different climate stats and zones for the different pathways.

In [19]:
def create_dataframe_period(data_period_scenario, period, scenario):
    print(f'processing period {period} scenario {scenario}')
    df_mean = data_period_scenario['climate_mean'].to_dataframe().unstack(['time'])
    df_mean.columns = ['_'.join(map(str,l1))+ '_mean' for l1 in df_mean.columns]
    df_mean = df_mean.reset_index()
    
    df_std = data_period_scenario['climate_std'].to_dataframe().unstack(['time'])
    df_std.columns = ['_'.join(map(str,l1)) + '_std' for l1 in df_std.columns]
    df_std = df_std.reset_index()
    
    df_zone = data_period_scenario['climate_zone'].to_dataframe()
    df_zone = df_zone.reset_index()

    df_climate_zones = df_mean.merge(df_std, on=['lat','lon'])
    df_climate_zones = df_climate_zones.merge(df_zone, on=['lat','lon'])
    
    df_climate_zones = df_climate_zones[~df_climate_zones['precipitation_1.0_mean'].isna()]
    df_climate_zones['period_start'] = period[0]
    df_climate_zones['period_end'] = period[1] 
    df_climate_zones['scenario'] = scenario

    df_climate_zones['climate_group'] = df_climate_zones['kg_class'].apply(lambda v1: climate_group_lookup[int(v1)])
    df_climate_zones['climate_subgroup'] = df_climate_zones['kg_class'].apply(lambda v1: climate_group_lookup[int(v1)])
    
    return df_climate_zones

In [20]:
def process_period_scenario(path_dict, current_res, current_period, current_scenario, out_path):
    current_data = {ds_id: xarray.open_dataset(current_path[current_res]) for ds_id, current_path in path_dict.items()}
    # process into a dataframe
    current_df = create_dataframe_period(
        current_data, 
        current_period,
        current_scenario)
    current_df.to_csv(out_path, index=False)
    return None
    

In [21]:
import multiprocessing

In [22]:
inter_paths = []
pool_args_list = {}
for current_res, res_str in resolutions_dict.items():
    climate_zones_df_list = []
    res_args = []
    for current_period, scenario_paths in data_path_dict.items():
        for current_scenario, scenario_data in scenario_paths.items():
            # load data for mthis poeriod/scenario
            # current_data = {ds_id: xarray.open_dataset() for ds_id, current_path in scenario_data.items()}
            # # process into a dataframe
            # climate_zones_df_list += [create_dataframe_period(
            #     current_data, 
            #     current_period,
            #     current_scenario)]
            out_path = ml_ready_output_dir / period_scenario_template.format(res=resolutions_dict[current_res],
                                                                             start=current_period[0],
                                                                             end=current_period[1],
                                                                             scenario=current_scenario,
                                                                            )
            inter_paths += [out_path]
            res_args  += [(scenario_data,
                                current_res,
                                current_period,
                                current_scenario,
                                out_path,
                               )]
    pool_args_list[current_res] = res_args



In [23]:
spice_pool = multiprocessing.Pool(2)
spice_pool

<multiprocessing.pool.Pool state=RUN pool_size=2>

processing period (1991, 2020) scenario historic
processing period (1901, 1930) scenario historic
processing period (1931, 1960) scenario historic
processing period (2041, 2070) scenario ssp119
processing period (1961, 1990) scenario historic
processing period (2041, 2070) scenario ssp126
processing period (2041, 2070) scenario ssp245
processing period (2041, 2070) scenario ssp460
processing period (2041, 2070) scenario ssp370
processing period (2041, 2070) scenario ssp585
processing period (2041, 2070) scenario ssp434
processing period (2071, 2099) scenario ssp119
processing period (2071, 2099) scenario ssp126
processing period (2071, 2099) scenario ssp434
processing period (2071, 2099) scenario ssp245
processing period (2071, 2099) scenario ssp460
processing period (2071, 2099) scenario ssp370
processing period (2071, 2099) scenario ssp585
processing period (2041, 2070) scenario ssp370
processing period (2041, 2070) scenario ssp585
processing period (2041, 2070) scenario ssp434
proce

In [2]:
! ls -l /data/users/stephen.haddad/climate_zones/ml_ready

total 44897032
-rw-r--r--. 1 stephen.haddad stephen.haddad 15941825848 Feb 20 10:16 climate_zones_0p1.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad 13936634539 Feb 19 17:31 climate_zones_0p5.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad   853179559 Feb 20 09:25 climate_zones_1901_1930_historic_0p1.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad    32709064 Feb 20 10:16 climate_zones_1901_1930_historic_0p5.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad     8287562 Feb 17 16:45 climate_zones_1901_1930_historic_1p0.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad   852941479 Feb 20 09:28 climate_zones_1931_1960_historic_0p1.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad    32697776 Feb 20 10:16 climate_zones_1931_1960_historic_0p5.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad     8285710 Feb 17 16:45 climate_zones_1931_1960_historic_1p0.csv
-rw-r--r--. 1 stephen.haddad stephen.haddad   853354293 Feb 20 09:31 climate_zones_1961_1990_historic_0p1.csv
-rw-r--r--. 1 stephen.haddad step

Now that we have set up all the parameter combinations, we can execute the parallel tasks for each of the resolutions in turn

In [ ]:
for current_res, res_str in resolutions_dict.items():
    print(current_res)
    res_it = spice_pool.starmap(process_period_scenario, pool_args_list[current_res])
    #trigger execution, but throw away data and instead read from disk, to reduce memory usage.
    _ = [_ for df_ps in res_it]
    climate_zones_merged_df = pandas.concat([pandas.read_csv(path1) for path1 in inter_paths]).reset_index().drop(['index'],axis='columns')
    # climate_zones_merged_df = pandas.concat([df_ps for df_ps in res_it]).reset_index().drop(['index'],axis='columns')
    out_path = ml_ready_output_dir / csv_out_template.format(resolution=resolutions_dict[current_res])
    print(out_path)
    climate_zones_merged_df.to_csv(out_path, index=False)


0.1
/data/users/stephen.haddad/climate_zones/ml_ready/climate_zones_0p1.csv


In [ ]:
current_res


### References
